# AEM4L2 E02 - Audio correcto vs audio problemático

**Objetivo:** entender la diferencia entre una transcripción correcta y una incorrecta antes de calcular métricas.

Este notebook construye la intuición: qué dijo la persona, qué escribió el ASR y por qué un error pequeño puede cambiar mucho.

**Python exercise relacionado:** `python_puro/AEM4_python_exercises/AEM4L2_audio_pipelines/e01_audio_transcripcion_basica.py`


## Tres nombres fundamentales

| Término | Definición | Ejemplo |
|---|---|---|
| Audio | señal original hablada | `llamada_soporte.wav` |
| Referencia | transcripción correcta validada | "pedido número cuatro cinco dos uno" |
| Hypothesis | salida del ASR | "pedido número cuatro cinco dos uno" o una versión con errores |

La referencia es el **ground truth**. La hypothesis es la predicción del modelo.


In [ ]:
reference = "Hola, quiero cancelar mi pedido urgente."
hypothesis_ok = "Hola, quiero cancelar mi pedido urgente."
hypothesis_bad = "Hola, quiero cambiar mi pedido urgente."

print('REFERENCIA:', reference)
print('ASR correcto:', hypothesis_ok)
print('ASR incorrecto:', hypothesis_bad)


## Diferencia semántica

`cancelar` y `cambiar` suenan parecidas, pero no significan lo mismo.

| Palabra esperada | Palabra ASR | Tipo de problema | Impacto |
|---|---|---|---|
| cancelar | cambiar | sustitución | el sistema ejecuta acción incorrecta |
| ocho | dos | sustitución | cambia una dosis médica |
| mil | millón | sustitución | cambia un monto financiero |
| no alérgico | alérgico | deleción de negación | cambia decisión clínica |

Frase docente: **no todos los errores pesan igual.**


In [ ]:
examples = [
    ("cancelar", "cambiar", "soporte", "acción equivocada"),
    ("ocho", "dos", "medicina", "frecuencia equivocada"),
    ("mil", "millón", "finanzas", "monto equivocado"),
    ("no", "", "medicina", "se pierde una negación"),
]

for expected, got, domain, impact in examples:
    print(f"{domain:<9} esperado={expected!r:<10} ASR={got!r:<10} impacto={impact}")


## Errores cotidianos de ASR

| Situación | Qué puede pasar | Ejemplo |
|---|---|---|
| Ruido de fondo | inserta palabras inexistentes | aparece "eh", "un", "the" |
| Habla rápida | omite palabras | se pierde un número de pedido |
| Acento o pronunciación | sustituye palabras | `pedido` -> `perdido` |
| Pausas raras | corta oraciones | cambia puntuación o sentido |
| Audio entrecortado | elimina fragmentos | falta media frase |
| Palabras técnicas | confunde jerga | `endpoint` -> `en punto` |


## Propagación del error

Un error temprano puede contaminar todo lo demás.

```text
Audio con ruido
  -> ASR confunde "cancelar" con "cambiar"
  -> LLM resume "quiere cambiar el pedido"
  -> backend crea ticket de modificación
  -> agente ejecuta acción equivocada
```

Eso se llama **error propagation**: el error nace en una etapa y se amplifica en las siguientes.


In [ ]:
def fake_pipeline(asr_text: str) -> dict:
    if "cancelar" in asr_text:
        intent = "cancellation"
        action = "cancelar pedido"
    elif "cambiar" in asr_text:
        intent = "change_request"
        action = "modificar pedido"
    else:
        intent = "unknown"
        action = "revisar manualmente"
    return {"asr_text": asr_text, "intent": intent, "action": action}

for text in [hypothesis_ok, hypothesis_bad]:
    print(fake_pipeline(text))


## Cómo usar las variantes de audio en clase

Cambiando solo `AUDIO_PATH`, podés comparar transcripciones reales:

| Archivo | Qué muestra |
|---|---|
| `llamada_soporte.wav` | audio base claro |
| `llamada_soporte_ruido.wav` | ruido de fondo |
| `llamada_soporte_rapido.wav` | habla acelerada |
| `llamada_soporte_entrecortado.wav` | microcortes |
| `llamada_soporte_pausas.wav` | pausas largas |
| `llamada_soporte_mal_estado.wav` | caso difícil combinado |

Pregunta para el aula:

> ¿Qué variante creen que va a producir más errores: ruido, rapidez, cortes o pausas?
